In [5]:
!pip install -q playwright pandas beautifulsoup4 lxml psutil
!python -m playwright install chromium

You should consider upgrading via the 'C:\Users\User\Documents\Yandex.Disk\Магистратура\2 курс\DataMining\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [12]:
# playwright install chromium
# playwright install-deps chromium

In [7]:
import asyncio, time, json, re, html, os, psutil, threading, sys
import pandas as pd
import requests
from pathlib import Path
from functools import wraps
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

URL = "https://scrapingcourse.com/button-click"
ECOMMERCE_BASE = "https://scrapingcourse.com/ecommerce/page/{}/"
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
NOC = Path("nocode"); NOC.mkdir(exist_ok=True)

LOG = []
MAX_ITERS, MAX_RECORDS, NO_GROWTH_LIMIT = 20, 220, 3
MAX_PAGES_LIGHT = 20


def run_async(coro_func):
    """Запускает async-функцию в отдельном потоке с ProactorEventLoop.
    Обходит NotImplementedError в Jupyter на Windows."""
    @wraps(coro_func)
    def wrapper(*args, **kwargs):
        result, exception = [], []

        def target():
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            try:
                result.append(loop.run_until_complete(coro_func(*args, **kwargs)))
            except Exception as e:
                exception.append(e)
            finally:
                loop.close()

        t = threading.Thread(target=target)
        t.start()
        t.join()
        if exception:
            raise exception[0]
        return result[0] if result else None

    return wrapper

print("Хелпер run_async готов")

Хелпер run_async готов


In [ ]:
@run_async
async def find_xhr():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        page = await (await browser.new_context()).new_page()
        xhrs = []
        def on_resp(r):
            if "json" in r.headers.get("content-type", "").lower():
                xhrs.append({"url": r.url, "status": r.status, "method": r.request.method})
        page.on("response", on_resp)
        await page.goto(URL, wait_until="networkidle")
        for _ in range(5):
            btn = page.locator("#load-more-btn")
            if await btn.count() == 0 or not await btn.is_visible(): break
            await btn.click()
            await page.wait_for_timeout(800)
        await browser.close()
        return xhrs


xhr_list = find_xhr()
print(f"JSON-ответов через XHR: {len(xhr_list)}")
for x in xhr_list[:5]:
    print(f"  {x['method']} {x['url']} — {x['status']}")

JSON-ответов через XHR: 0


In [10]:
print("=" * 60)
print("ПРОТОКОЛ ДИАГНОСТИКИ ИСТОЧНИКА")
print("=" * 60)

r = requests.get(URL, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(r.text, "html.parser")

n_prod = len(soup.select(".product"))
n_item = len(soup.select(".product-item"))
n_data = len(soup.select("[data-product-id]"))
n_ld   = len(re.findall(r'<script[^>]*type="application/(?:ld\+)?json"', r.text))
n_xhr  = len(re.findall(r'["\'](/[a-z\-/]*(?:api|load|product)[a-z\-/]*)["\']', r.text, re.I))
has_txt = "Chaz Kangeroo Hoodie" in r.text

print(f"""
1) requests:        HTTP {r.status_code}, {len(r.text)} байт
2) Селекторы:       .product={n_prod}, .product-item={n_item}, [data-product-id]={n_data}
3) JSON-LD:         {n_ld}
4) Целевой текст:   {has_txt}
5) XHR в HTML:      {n_xhr}
6) JS-off товаров:  {n_item}
""")
print("ВЫВОД:")
print(f"""
• Первая партия ({n_item} товаров) статична.
• Остальные — по клику «Load more» через JS.
• JSON-LD нет → вариант A не подходит.
• XHR в HTML нет → вариант B не подходит.
• Рабочая дешёвая альтернатива: вариант C.
""")

ПРОТОКОЛ ДИАГНОСТИКИ ИСТОЧНИКА

1) requests:        HTTP 200, 15556 байт
2) Селекторы:       .product=0, .product-item=12, [data-product-id]=0
3) JSON-LD:         0
4) Целевой текст:   True
5) XHR в HTML:      0
6) JS-off товаров:  12

ВЫВОД:

• Первая партия (12 товаров) статична.
• Остальные — по клику «Load more» через JS.
• JSON-LD нет → вариант A не подходит.
• XHR в HTML нет → вариант B не подходит.
• Рабочая дешёвая альтернатива: вариант C.



In [12]:
@run_async
async def collect_browser():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        ctx = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
            viewport={"width": 1920, "height": 1080}, locale="en-US")
        ctx.set_default_timeout(15000)
        ctx.set_default_navigation_timeout(30000)

        blocked = {"image": 0, "font": 0, "media": 0, "stylesheet": 0}
        async def block(route):
            rt = route.request.resource_type
            if rt in blocked:
                blocked[rt] += 1
                await route.abort()
            else:
                await route.continue_()
        await ctx.route("**/*", block)

        req, traffic = {"n": 0}, {"bytes": 0}
        page = await ctx.new_page()
        page.on("request", lambda r: req.__setitem__("n", req["n"] + 1))
        async def on_resp(resp):
            try: traffic["bytes"] += len(await resp.body())
            except Exception: pass
        page.on("response", lambda r: asyncio.create_task(on_resp(r)))

        await page.goto(URL, wait_until="domcontentloaded")
        SEL, BTN = ".product-item", "#load-more-btn"
        await page.wait_for_selector(SEL, timeout=15000)

        prev, no_growth, it = 0, 0, 0
        while it < MAX_ITERS and no_growth < NO_GROWTH_LIMIT:
            it += 1; t0 = time.time()
            try:
                btn = page.get_by_role("button", name=re.compile(r"load\s*more", re.I))
                if await btn.count() == 0:
                    btn = page.locator(BTN)
                if await btn.count() == 0 or not await btn.is_visible():
                    LOG.append({"mode":"browser","iteration":it,"records":prev,"growth":0,
                                "duration":round(time.time()-t0,2),"url":page.url,
                                "note":"кнопка недоступна"})
                    print(f"  [{it}] кнопка недоступна — конец данных"); break

                await btn.click()
                try:
                    await page.wait_for_function(
                        f"document.querySelectorAll('{SEL}').length > {prev}", timeout=8000)
                except Exception as e:
                    LOG.append({"mode":"browser","iteration":it,"records":prev,"growth":0,
                                "duration":round(time.time()-t0,2),"url":page.url,
                                "note":f"wait_for_function: {type(e).__name__}"})

                cur = await page.locator(SEL).count(); growth = cur - prev
                LOG.append({"mode":"browser","iteration":it,"records":cur,"growth":growth,
                            "duration":round(time.time()-t0,2),"url":page.url,"note":""})
                print(f"  [{it}] записей: {cur} (+{growth})")
                no_growth = no_growth + 1 if growth == 0 else 0
                prev = cur
                if cur >= MAX_RECORDS:
                    print(f"  достигнут лимит {MAX_RECORDS}"); break
                await page.wait_for_load_state("networkidle", timeout=2000)
            except Exception as e:
                await page.screenshot(path=ART / f"fail_iter{it}.png")
                (ART / f"fail_iter{it}.html").write_text(await page.content(), encoding="utf-8")
                LOG.append({"mode":"browser","iteration":it,"records":prev,"growth":0,
                            "duration":round(time.time()-t0,2),"url":page.url,
                            "note":f"ERROR: {e}"})
                print(f"  ❌ [{it}] {e}"); break

        t_ex = time.time()
        products = await page.eval_on_selector_all(SEL, """els => els.map(el => {
            const a = el.querySelector('a[href]');
            const n = el.querySelector('.product-name');
            const p = el.querySelector('.product-price');
            const i = el.querySelector('img');
            return { name: n?n.innerText.trim():null, price: p?p.innerText.trim():null,
                     url: a?a.href:null, image: i?i.getAttribute('src'):null };
        })""")
        extract_time = time.time() - t_ex
        await page.screenshot(path=ART / "success_page.png")
        await browser.close()
        return products[:MAX_RECORDS], dict(blocked), req["n"], extract_time, traffic["bytes"]/1024/1024


print("\n БРАУЗЕРНЫЙ СБОР")
t0 = time.perf_counter()
browser_data, blocked_res, req_count, extract_time, traffic_browser = collect_browser()
browser_time = time.perf_counter() - t0
print(f"\nЭлементов {len(browser_data)} за {browser_time:.2f}с | Запросов: {req_count} | Трафик: {traffic_browser:.2f} МБ")

pd.DataFrame(browser_data).to_csv("data_browser.csv", index=False, encoding="utf-8")
pd.DataFrame(LOG).to_csv("collection_log.csv", index=False, encoding="utf-8")


 БРАУЗЕРНЫЙ СБОР
  [1] записей: 12 (+12)
  [2] записей: 24 (+12)
  [3] записей: 36 (+12)
  [4] записей: 48 (+12)
  [5] записей: 60 (+12)
  [6] записей: 72 (+12)
  [7] записей: 84 (+12)
  [8] записей: 96 (+12)
  [9] записей: 120 (+24)
  [10] записей: 132 (+12)
  [11] записей: 144 (+12)
  [12] записей: 156 (+12)
  [13] записей: 168 (+12)
  [14] записей: 180 (+12)
  [15] записей: 192 (+12)
  [16] записей: 204 (+12)
  [17] записей: 216 (+12)
  [18] записей: 223 (+7)
  достигнут лимит 220

Элементов 220 за 5.91с | Запросов: 245 | Трафик: 0.96 МБ


In [13]:
@run_async
async def collect_light_variant_C():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox"])
        ctx = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0")
        ctx.set_default_timeout(15000)
        async def block_heavy(route):
            if route.request.resource_type in ("image","font","media","stylesheet"):
                await route.abort()
            else:
                await route.continue_()
        await ctx.route("**/*", block_heavy)

        req, traffic = {"n": 0}, {"bytes": 0}
        SEL, BTN = ".product-item", "#load-more-btn"
        page = await ctx.new_page()
        page.on("request", lambda r: req.__setitem__("n", req["n"] + 1))
        async def on_resp(resp):
            cl = resp.headers.get("content-length")
            if cl and cl.isdigit():
                traffic["bytes"] += int(cl)
        page.on("response", lambda r: asyncio.create_task(on_resp(r)))

        await page.goto(URL, wait_until="commit")
        await page.wait_for_selector(SEL, timeout=15000)

        prev = 0
        for i in range(MAX_ITERS):
            it = i + 1; t0 = time.time()
            btn = page.get_by_role("button", name=re.compile(r"load\s*more", re.I))
            if await btn.count() == 0:
                btn = page.locator(BTN)
            if await btn.count() == 0 or not await btn.is_visible():
                LOG.append({"mode":"light","iteration":it,"records":prev,"growth":0,
                            "duration":round(time.time()-t0,2),"url":page.url,
                            "note":"кнопка недоступна"})
                print(f"  [{it}] кнопка недоступна — конец данных"); break
            try:
                await btn.click()
                await page.wait_for_function(
                    f"document.querySelectorAll('{SEL}').length > {prev}",
                    timeout=1000, polling=50)
            except Exception as e:
                LOG.append({"mode":"light","iteration":it,"records":prev,"growth":0,
                            "duration":round(time.time()-t0,2),"url":page.url,
                            "note":f"wait_for_function: {type(e).__name__}"})
                print(f"  [{it}] нет прироста — стоп"); break

            cur = await page.locator(SEL).count(); growth = cur - prev
            LOG.append({"mode":"light","iteration":it,"records":cur,"growth":growth,
                        "duration":round(time.time()-t0,2),"url":page.url,"note":""})
            print(f"  [{it}] записей: {cur} (+{growth})")
            prev = cur
            if cur >= MAX_RECORDS:
                print(f"  достигнут лимит {MAX_RECORDS}"); break

        data = await page.eval_on_selector_all(SEL, """els => els.map(el => {
            const n = el.querySelector('.product-name');
            const p = el.querySelector('.product-price');
            const a = el.querySelector('a[href]');
            return { name: n?n.innerText.trim():null, price: p?p.innerText.trim():null,
                     url: a?a.href:null };
        })""")
        await browser.close()
        return data[:MAX_RECORDS], req["n"], traffic["bytes"]/1024/1024


print("\n ДЕШЁВАЯ АЛЬТЕРНАТИВА (вариант C)")
t0 = time.perf_counter()
light_data, req_count_light, traffic_light = collect_light_variant_C()
light_time = time.perf_counter() - t0
print(f"Элементов {len(light_data)} за {light_time:.2f}с | Запросов: {req_count_light} | Трафик: {traffic_light:.2f} МБ")

pd.DataFrame(light_data).to_csv("data_light.csv", index=False, encoding="utf-8")


 ДЕШЁВАЯ АЛЬТЕРНАТИВА (вариант C)
  [1] записей: 12 (+12)
  [2] записей: 36 (+24)
  [3] записей: 48 (+12)
  [4] записей: 60 (+12)
  [5] записей: 72 (+12)
  [6] записей: 84 (+12)
  [7] записей: 96 (+12)
  [8] записей: 108 (+12)
  [9] записей: 120 (+12)
  [10] записей: 132 (+12)
  [11] записей: 144 (+12)
  [12] записей: 156 (+12)
  [13] записей: 168 (+12)
  [14] записей: 180 (+12)
  [15] записей: 187 (+7)
  [16] нет прироста — стоп
Элементов 187 за 6.30с | Запросов: 217 | Трафик: 0.20 МБ


In [14]:
def collect_static(url, max_pages=MAX_PAGES_LIGHT):
    headers = {"User-Agent": "Mozilla/5.0 (educational bot)"}
    data, t0 = [], time.perf_counter()
    for page in range(1, max_pages):
        u = url.format(page) if "{}" in url else url
        try:
            r = requests.get(u, headers=headers, timeout=10)
            if r.status_code == 404:
                print(f"  {u}: 404 — стоп"); break
            if r.status_code != 200:
                print(f"  {u}: HTTP {r.status_code}"); break
            soup = BeautifulSoup(r.text, "html.parser")
            items = []
            for card in soup.select("li.product"):
                n = card.select_one(".woocommerce-loop-product__title, h2")
                p = card.select_one(".price")
                a = card.select_one("a.woocommerce-LoopProduct-link, a")
                if not n: continue
                items.append({"name": n.get_text(strip=True),
                              "price": p.get_text(strip=True) if p else None,
                              "url": a["href"] if a else None})
            if not items: break
            data.extend(items)
            print(f"  {u}: +{len(items)} (всего {len(data)})")
            time.sleep(0.4)
            if len(data) >= MAX_RECORDS: break
        except Exception as e:
            print(f"{u}: {e}"); break
    return data, time.perf_counter() - t0


print("\n КОНТРОЛЬНЫЙ ИСТОЧНИК /ecommerce/")
control_data, control_time = collect_static(ECOMMERCE_BASE, MAX_PAGES_LIGHT)
print(f"Элементов {len(control_data)} за {control_time:.2f}с")

pd.DataFrame(control_data).to_csv("data_control.csv", index=False, encoding="utf-8")


 КОНТРОЛЬНЫЙ ИСТОЧНИК /ecommerce/
  https://scrapingcourse.com/ecommerce/page/1/: +16 (всего 16)
  https://scrapingcourse.com/ecommerce/page/2/: +16 (всего 32)
  https://scrapingcourse.com/ecommerce/page/3/: +16 (всего 48)
  https://scrapingcourse.com/ecommerce/page/4/: +16 (всего 64)
  https://scrapingcourse.com/ecommerce/page/5/: +16 (всего 80)
  https://scrapingcourse.com/ecommerce/page/6/: +16 (всего 96)
  https://scrapingcourse.com/ecommerce/page/7/: +16 (всего 112)
  https://scrapingcourse.com/ecommerce/page/8/: +16 (всего 128)
  https://scrapingcourse.com/ecommerce/page/9/: +16 (всего 144)
  https://scrapingcourse.com/ecommerce/page/10/: +16 (всего 160)
  https://scrapingcourse.com/ecommerce/page/11/: +16 (всего 176)
  https://scrapingcourse.com/ecommerce/page/12/: +12 (всего 188)
  https://scrapingcourse.com/ecommerce/page/13/: 404 — стоп
Элементов 188 за 19.31с


In [15]:
b = pd.DataFrame(browser_data); b["source"] = "button-click"
l = pd.DataFrame(light_data);   l["source"] = "button-click (light)"
c = pd.DataFrame(control_data); c["source"] = "ecommerce"

final = pd.concat([b, l, c], ignore_index=True)
final = final.drop_duplicates(subset=["name", "url"]).reset_index(drop=True)
final.to_csv("data_browser.csv", index=False, encoding="utf-8")

print(f"Финальный набор: {len(final)} записей")
print(final["source"].value_counts().to_string())

Финальный набор: 335 записей
source
ecommerce               188
button-click            144
button-click (light)      3


In [16]:
def norm(s):
    if pd.isna(s): return ""
    return re.sub(r"\s+", " ", html.unescape(str(s))).strip().lower()

bs = {norm(n) for n in b["name"]} - {""}
ls = {norm(n) for n in l["name"]} - {""}
cs = {norm(n) for n in c["name"]} - {""}
common, only_b, only_l = bs & ls, bs - ls, ls - bs

print("=" * 60); print("СВЕРКА НАБОРОВ"); print("=" * 60)
print(f"  Browser: {len(bs)} | Light: {len(ls)} | Control: {len(cs)}")
print(f"  Browser ∩ Light: {len(common)} ({len(common)/max(len(bs),1)*100:.1f}%)")
print(f"  Только browser: {len(only_b)} | Только light: {len(only_l)}")
if only_l:
    print(f"\n  Только в light ({len(only_l)}):")
    for n in sorted(only_l)[:10]: print(f"    • {n}")
print(f"""
  ИНТЕРПРЕТАЦИЯ:
    • Browser и Light — один источник (/button-click).
    • Control — контрольный /ecommerce/ для сверки объёма.
    • Расхождения с control — товары вне /button-click.
""")

СВЕРКА НАБОРОВ
  Browser: 144 | Light: 147 | Control: 182
  Browser ∩ Light: 144 (100.0%)
  Только browser: 0 | Только light: 3

  Только в light (3):
    • antonia racer tank
    • breathe-easy tank
    • maya tunic

  ИНТЕРПРЕТАЦИЯ:
    • Browser и Light — один источник (/button-click).
    • Control — контрольный /ecommerce/ для сверки объёма.
    • Расхождения с control — товары вне /button-click.



In [17]:
df = pd.DataFrame(browser_data)
print("=" * 60); print("КАЧЕСТВО ДАННЫХ"); print("=" * 60)
print(f"\nВсего: {len(df)} | Уникальных url: {df['url'].nunique()} | "
      f"Дубли: {df['url'].duplicated().sum()} ({df['url'].duplicated().sum()/len(df)*100:.1f}%)")

print("\nПолнота ключевых полей:")
for col in ["name","price","url"]:
    print(f"  {col:8s}: {df[col].notna().mean()*100:6.1f}%")

df["price_num"] = (df["price"].astype(str).str.replace(r"[^\d.]","",regex=True)
                   .replace("", None).astype(float, errors="ignore"))
print(f"\nprice_num: {df['price_num'].dtype}, min={df['price_num'].min():.2f}, max={df['price_num'].max():.2f}")

print("\nБизнес-проверки:")
errors = 0
for cond, label in [
    ((df["price_num"].dropna() <= 0).sum(), "price <= 0"),
    ((~df["url"].dropna().str.startswith("http")).sum(), "url без http"),
    ((df["name"].dropna().str.len() < 2).sum(), "name < 2")]:
    print(f"  {label}: {cond}"); errors += cond
print(f"  → нарушений: {errors}")

def smoke_check(df, cols=("name","price"), threshold=0.05):
    for col in cols:
        if df[col].isna().mean() > threshold:
            print(f"{col}: {df[col].isna().mean()*100:.1f}% пустых"); return
    print(f"\nSmoke-check пройден (порог {threshold*100}%)")
smoke_check(df)

КАЧЕСТВО ДАННЫХ

Всего: 220 | Уникальных url: 144 | Дубли: 76 (34.5%)

Полнота ключевых полей:
  name    :  100.0%
  price   :  100.0%
  url     :  100.0%

price_num: float64, min=18.00, max=99.00

Бизнес-проверки:
  price <= 0: 0
  url без http: 0
  name < 2: 0
  → нарушений: 0

Smoke-check пройден (порог 5.0%)
